In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import RFE, SelectKBest, f_regression

In [ ]:
rng = np.random.default_rng(42)
n = 200

In [ ]:
area      = rng.normal(1800, 500, n).clip(600, 4000).round(0)
bedrooms  = np.clip((area / 600 + rng.normal(0, 0.7, n)).round(), 1, 6)
age       = rng.uniform(0, 50, n).round(1)
distance  = rng.uniform(1, 30, n).round(1)
location  = rng.choice(["Downtown", "Suburb", "Rural"], size=n, p=[0.3, 0.4, 0.3])
loc_effect = pd.Series(location).map({"Downtown": 0, "Suburb": -15000, "Rural": -40000}).values

In [ ]:
price = (50000 + 120 * area + 8000 * bedrooms
         - 900 * age - 2500 * distance + loc_effect
         + rng.normal(0, 20000, n)).round(2)

In [ ]:
data = pd.DataFrame({
    "Area_sqft": area,
    "Bedrooms": bedrooms,
    "Age_years": age,
    "Distance_to_City_km": distance,
    "Location": location,
    "Price": price
})

In [ ]:
for col in ["Area_sqft", "Age_years", "Location"]:
    idx = rng.choice(n, size=int(0.05 * n), replace=False)
    data.loc[idx, col] = np.nan

In [ ]:
data.to_csv("house_prices.csv", index=False)
print("CSV saved as house_prices.csv")

In [ ]:
dataset = pd.read_csv("house_prices.csv")
print("\nShape (rows, columns):", dataset.shape)
print("\nFirst 5 rows:\n", dataset.head())
print("\nInfo:")
dataset.info()
print("\nSummary statistics:\n", dataset.describe(include="all"))
print("\nMissing values:\n", dataset.isnull().sum())
print("\nLocation counts:\n", dataset["Location"].value_counts(dropna=False))

In [ ]:
dataset.hist(bins=20, figsize=(10, 7), edgecolor="black")
plt.suptitle("Distribution of Numeric Variables")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Location", data=dataset, order=["Downtown", "Suburb", "Rural"])
plt.title("Distribution of Location")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x="Location", y="Price", data=dataset, order=["Downtown", "Suburb", "Rural"])
plt.title("Price by Location")
plt.show()

In [ ]:
corr = dataset.corr(numeric_only=True)
print("\nCorrelation matrix:\n", corr)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = dataset.drop("Price", axis=1)
y = dataset["Price"]

In [ ]:
num_cols = ["Area_sqft", "Bedrooms", "Age_years", "Distance_to_City_km"]
cat_cols = ["Location"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

In [ ]:
print("\nX_train:", x_train.shape, " X_test:", x_test.shape)
print("y_train:", y_train.shape, " y_test:", y_test.shape)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(drop="first", sparse_output=False))
        ]), cat_cols)
    ],
    verbose_feature_names_out=False
)
preprocess.set_output(transform="pandas")

In [ ]:
x_train_imp = preprocess.fit_transform(x_train)
x_test_imp  = preprocess.transform(x_test)
feature_names = x_train_imp.columns

In [ ]:
print("\nColumns after one-hot encoding:", list(feature_names))
print(x_train_imp.head())

In [ ]:
scaler = StandardScaler()
x_train_sc = pd.DataFrame(scaler.fit_transform(x_train_imp), columns=feature_names, index=x_train_imp.index)
x_test_sc  = pd.DataFrame(scaler.transform(x_test_imp), columns=feature_names, index=x_test_imp.index)

In [ ]:
model = LinearRegression()
model.fit(x_train_imp, y_train)
print("\nModel has been trained successfully")

In [ ]:
y_pred = model.predict(x_test_imp)

In [ ]:
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
n_test, p = x_test_imp.shape
adj_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - p - 1)

In [ ]:
print("\n------ Model Performance on Test Set ------")
print("Train R2 score      :", model.score(x_train_imp, y_train))
print("Test R2 score       :", r2)
print("Adjusted R2         :", adj_r2)
print("Mean Squared Error  :", mse)
print("Root MSE            :", rmse)
print("Mean Absolute Error :", mae)

In [ ]:
pred_df = pd.DataFrame({"Actual Value": y_test.values,
                        "Predicted Value": y_pred,
                        "Difference": y_test.values - y_pred})
print("\n", pred_df.head(10))

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, color="blue", alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", label="Perfect fit")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.regplot(x=y_test.values, y=y_pred, color="red", ci=None)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Regression Plot of Model")
plt.show()

In [ ]:
residuals = y_test.values - y_pred
plt.figure(figsize=(6, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), num_cols):
    ax.scatter(x_train_imp[col], y_train, alpha=0.6)
    m, b = np.polyfit(x_train_imp[col], y_train, 1)
    xs = np.linspace(x_train_imp[col].min(), x_train_imp[col].max(), 100)
    ax.plot(xs, m * xs + b, color="red")
    ax.set_xlabel(col)
    ax.set_ylabel("Price")
    ax.set_title(f"{col} vs Price")
plt.tight_layout()
plt.show()

In [ ]:
print("\nIntercept:", model.intercept_)
coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": model.coef_})
print("\nCoefficients (original units):\n", coef_df)

In [ ]:
x_train_sm = sm.add_constant(x_train_imp)
ols_model = sm.OLS(y_train, x_train_sm).fit()
print("\n", ols_model.summary())

In [ ]:
scaled_model = LinearRegression().fit(x_train_sc, y_train)
importance = pd.DataFrame({
    "Feature": feature_names,
    "Standardized Coef": scaled_model.coef_,
    "Abs Importance": np.abs(scaled_model.coef_)
}).sort_values("Abs Importance", ascending=False)
print("\nFeature influence (standardized coefficients):\n", importance)

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(importance["Feature"], importance["Abs Importance"], color="teal")
plt.xlabel("Absolute Standardized Coefficient")
plt.title("Most Influential Predictors")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
new_data = pd.DataFrame({
    "Area_sqft": [1500, 2200, 3000],
    "Bedrooms": [3, 4, 5],
    "Age_years": [10, 5, 2],
    "Distance_to_City_km": [12, 8, 3],
    "Location": ["Rural", "Suburb", "Downtown"]
})
new_pred = model.predict(preprocess.transform(new_data))
new_data["Predicted_Price"] = new_pred.round(2)
print("\nPredictions on new data:\n", new_data)

In [ ]:
rfe = RFE(LinearRegression(), n_features_to_select=3)
rfe.fit(x_train_sc, y_train)
rfe_features = feature_names[rfe.support_]
print("\nRFE selected features:", list(rfe_features))
print("RFE ranking:", dict(zip(feature_names, rfe.ranking_)))

In [ ]:
skb = SelectKBest(score_func=f_regression, k=3)
skb.fit(x_train_imp, y_train)
skb_features = feature_names[skb.get_support()]
print("\nSelectKBest selected features:", list(skb_features))
print("F-scores:", dict(zip(feature_names, skb.scores_.round(2))))

In [ ]:
def evaluate(cols):
    m = LinearRegression().fit(x_train_imp[cols], y_train)
    pred = m.predict(x_test_imp[cols])
    return r2_score(y_test, pred), np.sqrt(mean_squared_error(y_test, pred))

In [ ]:
r2_full, rmse_full = evaluate(list(feature_names))
r2_rfe, rmse_rfe   = evaluate(list(rfe_features))
r2_skb, rmse_skb   = evaluate(list(skb_features))

In [ ]:
comparison = pd.DataFrame({
    "Model": ["All features", "RFE (3 features)", "SelectKBest (3 features)"],
    "R2": [r2_full, r2_rfe, r2_skb],
    "RMSE": [rmse_full, rmse_rfe, rmse_skb]
})
print("\nModel comparison:\n", comparison)

In [ ]:
cols = list(feature_names)
while len(cols) > 1:
    ols_tmp = sm.OLS(y_train, sm.add_constant(x_train_imp[cols])).fit()
    pvals = ols_tmp.pvalues.drop("const")
    if pvals.max() > 0.05:
        drop = pvals.idxmax()
        print(f"Removing '{drop}' (p-value = {pvals.max():.4f})")
        cols.remove(drop)
    else:
        break
print("Features kept after backward elimination:", cols)